# T1 — PMH on frozen features + classical ML

**Lemma D1** · `nuisance="subspace"` · [Task doc](../../docs/tasks/t01-classical.md) · Paper: [main.pdf](../../main.pdf)

Same **§1–8 spine** as other task notebooks; T1 adds **sub-experiments** below (Office-31, soft k-NN, ridge, MNIST drift).

| § | T1 content |
|---|------------|
| 1–4 | Install → sklearn demo data → scope → core loop + falsification |
| 5–8 | Office-31 · soft k-NN · ridge · MNIST drift |
| 9 | Your frozen-feature pipeline |

In [ ]:
!pip install -q "matching-pmh[sklearn,vision]"

## 1 — Plug in data (every classical T1 experiment)

| Slot | Meaning |
|------|--------|
| `x_source`, `y_source` | Training site — frozen `[N, d]` + labels |
| `x_target`, `y_target` | Production site — **same label semantics**, same `d` |

MNIST pixels, Office-31 ResNet-512, tabular rows → all become these four arrays.

In [ ]:
import numpy as np
from pmh import load_g2_demo_arrays

USE_DEMO = True
SEED = 0
RANK = 16

if USE_DEMO:
    x_source, y_source, x_target, y_target = load_g2_demo_arrays(n=800, seed=SEED)
else:
    x_source = np.load("YOUR_SITE_A/features.npy")
    y_source = np.load("YOUR_SITE_A/labels.npy")
    x_target = np.load("YOUR_SITE_B/features.npy")
    y_target = np.load("YOUR_SITE_B/labels.npy")

print("source", x_source.shape, "target", x_target.shape)

## 2 — Estimate \(\hat W\) once (`subspace` / D1)

In [ ]:
from pmh import PMHMatcher, suggest_nuisance
from pmh.sklearn_match import MatchedSubspaceProjector

print(suggest_nuisance(has_source_labels=True, has_target_labels=True, has_target_domain=True))

matcher = PMHMatcher(nuisance="subspace", rank=RANK, seed=SEED)
matcher.fit(x_source, y_source, x_target, y_target)
print("preflight", matcher.artifact_.preflight, "eigengap", matcher.artifact_.eigengap)

projector = MatchedSubspaceProjector(rank=RANK, seed=SEED).fit(x_source, y_source, x_target, y_target)
W_hat = projector.w_
x_src_pmh = projector.transform(x_source)
x_tgt_pmh = projector.transform(x_target)

## 3 — Three classical heads, same \(\hat W\) (logistic / SVM / k-NN)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier

_, x_te, _, y_te = train_test_split(x_target, y_target, test_size=0.35, random_state=SEED)

def acc(factory, xtr, ytr, xte, yte):
    c = factory()
    c.fit(xtr, ytr)
    return float(accuracy_score(yte, c.predict(xte)))

heads = {
    "logistic": lambda: LogisticRegression(max_iter=3000, random_state=SEED),
    "svm": lambda: LinearSVC(max_iter=5000, random_state=SEED),
    "knn": lambda: KNeighborsClassifier(n_neighbors=5),
}
print(f"{'head':10s}  {'B0':>8s}  {'PMH hard':>8s}  {'delta':>8s}")
for name, fac in heads.items():
    b0 = acc(fac, x_source, y_source, x_te, y_te)
    pmh = acc(fac, x_src_pmh, y_source, projector.transform(x_te), y_te)
    print(f"{name:10s}  {b0:8.3f}  {pmh:8.3f}  {pmh - b0:+8.3f}")

## 4 — Falsification + CORAL (`compare_arms_sklearn`, T1 protocol)

In [ ]:
from pmh import compare_arms_sklearn

def run_table(factory, label):
    res = compare_arms_sklearn(
        x_source, y_source, x_target, y_target,
        rank=RANK,
        classifier_factory=factory,
        preset="t1_synthetic_sklearn",
        include_coral=True,
        seed=SEED,
    )
    print(f"\n=== {label} ===")
    for arm, run in sorted(res.arms.items()):
        print(f"  {arm:12s}  {run.val_metric:.3f}")

run_table(lambda: LogisticRegression(max_iter=3000, random_state=SEED), "logistic")
run_table(lambda: LinearSVC(max_iter=5000, random_state=SEED), "linear SVM")

## 5 — Office-31 (Amazon → DSLR, ResNet-18, paper §9)

Set `RUN_OFFICE31 = True` after download (~80MB + torch). Same loop: extract features → `preset="t1_office31_sklearn"` (rank 32, pool 200, test 250).

In [ ]:
RUN_OFFICE31 = False
OFFICE31_ROOT = "./data/office31"

if RUN_OFFICE31:
    from pmh.datasets.office31 import extract_office31_features

    print("Extracting ResNet-18 features (may take a few minutes)...")
    x_o_src, y_o_src = extract_office31_features(OFFICE31_ROOT, "amazon", max_samples=2000, seed=SEED)
    x_o_tgt, y_o_tgt = extract_office31_features(OFFICE31_ROOT, "dslr", max_samples=2000, seed=SEED + 1)
    O_RANK = 32
    for label, fac in [
        ("logistic", lambda: LogisticRegression(max_iter=3000, random_state=SEED)),
        ("svm", lambda: LinearSVC(max_iter=5000, random_state=SEED)),
        ("knn", lambda: KNeighborsClassifier(n_neighbors=5)),
    ]:
        res = compare_arms_sklearn(
            x_o_src, y_o_src, x_o_tgt, y_o_tgt,
            rank=O_RANK,
            classifier_factory=fac,
            preset="t1_office31_sklearn",
            include_coral=True,
            seed=SEED,
        )
        print(f"\n=== Office-31 {label} ===")
        for arm, run in sorted(res.arms.items()):
            print(f"  {arm:12s}  {run.val_metric:.3f}")
else:
    print("Skip Office-31. Download then set RUN_OFFICE31=True:")
    print("  python scripts/download_office31.py --root", OFFICE31_ROOT)
    print("  python scripts/demos/office31_sklearn.py --office31-root", OFFICE31_ROOT)

## 6 — Soft k-NN (paper `soft_knn.py`, FINAL §3)

Hard \(P_{W^\perp}\) can **hurt** k-NN. **Soft lift** \(L = \sqrt{\beta} P_\perp + \sqrt{\alpha} P_W\) fixes it (CV on \(\alpha\)).

In [ ]:
from pmh.classical import compare_knn_hard_vs_soft

knn_arms = compare_knn_hard_vs_soft(x_source, y_source, x_target, y_target, W_hat, seed=SEED)
print("k-NN target accuracy:")
for k, v in knn_arms.items():
    if isinstance(v, float):
        print(f"  {k:14s}  {v:.3f}")
    else:
        print(f"  {k:14s}  {v}")

## 7 — Ridge on tabular (paper `ridge_tabular.py`, FINAL §7)

Real UCI features + structured nuisance. Same **project** then **Ridge** — MSE on target OOD.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

X, y = fetch_california_housing(return_X_y=True)
X = StandardScaler().fit_transform(X).astype(np.float32)
y = ((y - y.mean()) / y.std()).astype(np.float32)
rng = np.random.default_rng(SEED)
idx = rng.permutation(len(X))
n_tr, n_te = 4000, 2000
X_tr, y_tr = X[idx[:n_tr]], y[idx[:n_tr]]
X_te, y_te = X[idx[n_tr : n_tr + n_te]], y[idx[n_tr : n_tr + n_te]]

rank_t = 4
w_tab = rng.standard_normal((X.shape[1], rank_t)).astype(np.float32)
w_ls = np.linalg.lstsq(X_tr, y_tr, rcond=None)[0]
v = w_ls / (np.linalg.norm(w_ls) + 1e-12)
w_tab = w_tab - np.outer(v, v @ w_tab)
Q, _ = np.linalg.qr(w_tab)
W_tab = Q[:, :rank_t].astype(np.float32)
sigma = 0.5
X_tr_n = X_tr + sigma * ((X_tr @ W_tab) @ W_tab.T) + 0.05 * rng.standard_normal(X_tr.shape).astype(np.float32)
X_te_n = X_te + sigma * ((X_te @ W_tab) @ W_tab.T)

proj_tab = MatchedSubspaceProjector(rank=rank_t, seed=SEED).fit(X_tr, y_tr, X_te_n, y_te)

def ridge_mse(xtr, ytr, xte, yte):
    r = Ridge(alpha=1e-6)
    r.fit(xtr, ytr)
    return float(mean_squared_error(yte, r.predict(xte)))

print(f"Ridge B0 (nuisanced):     {ridge_mse(X_tr_n, y_tr, X_te_n, y_te):.4f}")
print(f"Ridge PMH (projected):  {ridge_mse(proj_tab.transform(X_tr_n), y_tr, proj_tab.transform(X_te_n), y_te):.4f}")

## 8 — MNIST pixels + data-driven \(W\) (paper `mnist_drift.py`, FINAL §4)

Small subset via OpenML — **estimated** \(\hat W\) (not oracle DCT). SVM four-arm table.

In [ ]:
try:
    from sklearn.datasets import fetch_openml

    X_m, y_m = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False, parser="auto")
    X_m = (X_m.astype(np.float32) / 255.0)
    y_m = y_m.astype(np.int64)
    n_tr, n_te = 2000, 1000
    rng_m = np.random.default_rng(SEED + 7)
    idx = rng_m.permutation(len(X_m))
    X_a = X_m[idx[:n_tr]]
    y_a = y_m[idx[:n_tr]]
    X_hold = X_m[idx[n_tr : n_tr + n_te]]
    y_hold = y_m[idx[n_tr : n_tr + n_te]]
    w_m = MatchedSubspaceProjector(rank=16, seed=SEED + 7).fit(X_a, y_a, X_hold, y_hold).w_
    nu = (X_a @ w_m) @ w_m.T
    X_d = X_a + 1.2 * nu + 0.05 * rng_m.standard_normal(X_a.shape).astype(np.float32)
    res_m = compare_arms_sklearn(
        X_a, y_a, X_d, y_hold,
        rank=16,
        classifier_factory=lambda: LinearSVC(max_iter=3000, random_state=SEED),
        preset="t1_synthetic_sklearn",
        include_coral=False,
        seed=SEED,
    )
    print("MNIST-style SVM (data-driven W, shifted deploy features):")
    for arm, run in sorted(res_m.arms.items()):
        print(f"  {arm:12s}  {run.val_metric:.3f}")
except Exception as e:
    print("MNIST section skipped:", e)
    print("Install/openml or see main.pdf T1 for full MNIST drift experiments")

## 9 — Your pipeline (copy-paste)

```python
from pmh import compare_arms_sklearn
from sklearn.svm import LinearSVC

compare_arms_sklearn(
    x_source, y_source, x_target, y_target,
    rank=32,
    classifier_factory=lambda: LinearSVC(),
    preset="t1_office31_sklearn",
    include_coral=True,
)
```

k-NN production head → `from pmh.classical import compare_knn_hard_vs_soft`

CLI: `pmh-train evaluate --source-dir site_a --target-dir site_b`